<a href="https://colab.research.google.com/github/danhhuit/image-captioning-distillation/blob/main/Tuan02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUẦN 02 - NGUYỄN THÀNH DANH

# 1. Bật GPU

In [17]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


# 2. Mount Drive

In [18]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Corrected path: Assumes 'KLCN' is a folder directly under 'My Drive'
PROJECT = Path("/content/drive/MyDrive/KLCN/NguyenThanhDanh")
DATA_DIR = PROJECT / "data"
CHECKPOINT_DIR = PROJECT / "checkpoints"
CONFIG_DIR = PROJECT / "configs"
FEATURE_DIR = PROJECT / "features"
LOG_DIR = PROJECT / "logs"
METRIC_DIR = PROJECT / "metrics"
PREDICTION_DIR = PROJECT / "predictions"
FIGURE_DIR = PROJECT / "figures"

for directory in [
    CHECKPOINT_DIR, CONFIG_DIR, FEATURE_DIR, LOG_DIR,
    METRIC_DIR, PREDICTION_DIR, FIGURE_DIR
]:
    directory.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 3. Import thư viện

In [19]:
!pip -q install pycocoevalcap
!pip -q install git+https://github.com/openai/CLIP.git

  Preparing metadata (setup.py) ... done


In [20]:
import json
import platform
import torch
import torchvision
import pandas as pd
import numpy as np

environment = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
}

print(json.dumps(environment, indent=2, ensure_ascii=False))

with open(CONFIG_DIR / "environment.json", "w", encoding="utf-8") as f:
    json.dump(environment, f, indent=2, ensure_ascii=False)

{
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "cuda_available": true,
  "gpu": "Tesla T4"
}


# 4. Chuẩn bị Flickr8K

4.1. Giải nén vào ổ tạm Colab

In [21]:
import zipfile
from pathlib import Path

ZIP_PATH = DATA_DIR / "Flickr8k.zip"
WORK_DIR = Path("/content/flickr8k")

assert ZIP_PATH.exists(), f"Không tìm thấy: {ZIP_PATH}"

if not WORK_DIR.exists():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(WORK_DIR)

jpg_files = list(WORK_DIR.rglob("*.jpg"))
print("Số file ảnh tìm thấy:", len(jpg_files))
print("Ví dụ:", jpg_files[:3])

Số file ảnh tìm thấy: 8091
Ví dụ: [PosixPath('/content/flickr8k/Images/2266144051_614b2d62b0.jpg'), PosixPath('/content/flickr8k/Images/3432495898_a5859f06b6.jpg'), PosixPath('/content/flickr8k/Images/3170110692_d1e0e66cee.jpg')]


4.2. Đọc caption

In [22]:
caption_file = next(WORK_DIR.rglob("captions.txt"))
captions_df = pd.read_csv(caption_file)

captions_df.columns = [c.strip().lower() for c in captions_df.columns]
captions_df = captions_df.rename(
    columns={"image_name": "image", "comment": "caption"}
)

captions_df["image"] = captions_df["image"].astype(str).str.strip()
captions_df["caption"] = captions_df["caption"].astype(str).str.strip()

print(captions_df.head())
print("Số dòng caption:", len(captions_df))
print("Số ảnh có caption:", captions_df["image"].nunique())

                       image  \
0  1000268201_693b08cb0e.jpg   
1  1000268201_693b08cb0e.jpg   
2  1000268201_693b08cb0e.jpg   
3  1000268201_693b08cb0e.jpg   
4  1000268201_693b08cb0e.jpg   

                                             caption  
0  A child in a pink dress is climbing up a set o...  
1              A girl going into a wooden building .  
2   A little girl climbing into a wooden playhouse .  
3  A little girl climbing the stairs to her playh...  
4  A little girl in a pink dress going into a woo...  
Số dòng caption: 40455
Số ảnh có caption: 8091


4.3. Chia train / val / test

In [23]:
def read_split(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_file = list(WORK_DIR.rglob("Flickr_8k.trainImages.txt"))
val_file = list(WORK_DIR.rglob("Flickr_8k.devImages.txt"))
test_file = list(WORK_DIR.rglob("Flickr_8k.testImages.txt"))

if train_file and val_file and test_file:
    train_images = read_split(train_file[0])
    val_images = read_split(val_file[0])
    test_images = read_split(test_file[0])
    split_method = "Flickr8K provided split files"
else:
    all_images = sorted(captions_df["image"].unique())

    rng = np.random.default_rng(42)
    rng.shuffle(all_images)

    train_images = all_images[:6000]
    val_images = all_images[6000:7000]
    test_images = all_images[7000:8000]
    split_method = "Deterministic random split, seed=42"

print("Train:", len(train_images))
print("Validation:", len(val_images))
print("Test:", len(test_images))
print("Split method:", split_method)

Train: 6000
Validation: 1000
Test: 1000
Split method: Deterministic random split, seed=42


In [24]:
splits = {
    "method": split_method,
    "seed": 42,
    "train": list(train_images),
    "validation": list(val_images),
    "test": list(test_images)
}

with open(CONFIG_DIR / "splits.json", "w", encoding="utf-8") as f:
    json.dump(splits, f, indent=2, ensure_ascii=False)

# 5. Xây dựng Vocabulary

In [25]:
import re
from collections import Counter

PAD_TOKEN = "<pad>"
BOS_TOKEN = "<bos>"
EOS_TOKEN = "<eos>"
UNK_TOKEN = "<unk>"

MIN_FREQ = 5
MAX_LEN = 30

def tokenize(text):
    text = text.lower().strip()
    return re.findall(r"[a-z0-9]+(?:'[a-z]+)?", text)

train_caption_df = captions_df[
    captions_df["image"].isin(train_images)
].copy()

counter = Counter()
for caption in train_caption_df["caption"]:
    counter.update(tokenize(caption))

itos = [PAD_TOKEN, BOS_TOKEN, EOS_TOKEN, UNK_TOKEN]
itos += sorted([
    token for token, frequency in counter.items()
    if frequency >= MIN_FREQ
])

stoi = {token: index for index, token in enumerate(itos)}

PAD_ID = stoi[PAD_TOKEN]
BOS_ID = stoi[BOS_TOKEN]
EOS_ID = stoi[EOS_TOKEN]
UNK_ID = stoi[UNK_TOKEN]

print("Vocabulary size:", len(itos))

Vocabulary size: 2556


In [26]:
vocab_data = {
    "min_freq": MIN_FREQ,
    "max_len": MAX_LEN,
    "itos": itos,
    "stoi": stoi
}

with open(CONFIG_DIR / "vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(vocab_data, f, indent=2, ensure_ascii=False)

# 6. Tổ chức Visual Encoder

In [27]:
from torch import nn
from torchvision.models import (
    resnet50,
    ResNet50_Weights,
    efficientnet_b3,
    EfficientNet_B3_Weights,
    vit_b_16,
    ViT_B_16_Weights
)
import clip

def build_encoder(model_name, device):
    if model_name == "resnet50":
        weights = ResNet50_Weights.IMAGENET1K_V2
        encoder = resnet50(weights=weights)
        encoder.fc = nn.Identity()
        transform = weights.transforms()
        feature_dim = 2048
        weight_name = "ResNet50 IMAGENET1K_V2"

    elif model_name == "efficientnet_b3":
        weights = EfficientNet_B3_Weights.IMAGENET1K_V1
        encoder = efficientnet_b3(weights=weights)
        encoder.classifier = nn.Identity()
        transform = weights.transforms()
        feature_dim = 1536
        weight_name = "EfficientNet-B3 IMAGENET1K_V1"

    elif model_name == "vit_b_16":
        weights = ViT_B_16_Weights.IMAGENET1K_V1
        encoder = vit_b_16(weights=weights)
        encoder.heads = nn.Identity()
        transform = weights.transforms()
        feature_dim = 768
        weight_name = "ViT-B/16 IMAGENET1K_V1"

    elif model_name == "clip_vit_b16":
        clip_model, transform = clip.load(
            "ViT-B/16",
            device="cpu",
            jit=False
        )
        encoder = clip_model.visual
        feature_dim = 512
        weight_name = "OpenAI CLIP ViT-B/16"

    else:
        raise ValueError(f"Model không hợp lệ: {model_name}")

    encoder.eval()

    for parameter in encoder.parameters():
        parameter.requires_grad = False

    encoder = encoder.to(device)
    return encoder, transform, feature_dim, weight_name

In [28]:
# Model dùng để chạy
MODEL_TO_RUN = "resnet50"

In [29]:
# Tổng hợp model
MODEL_TO_RUN = "efficientnet_b3"
MODEL_TO_RUN = "vit_b_16"
MODEL_TO_RUN = "clip_vit_b16"

In [30]:
# Checkpoint:
checkpoint = {
    "model_name": MODEL_TO_RUN,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,
    "vocabulary": vocab_data,
    "model_config": model_config,
    "environment": environment
}

NameError: name 'model' is not defined